In [ ]:
!jupyter nbconvert --to script pangeo-fish-unification_V2.ipynb

# Tutorial: how to use `pangeo-fish`


**Overview.**

This Jupyter notebook demonstrates how to use `pangeo-fish`.

Specifically, we will fit the geolocation on the data from the study conducted by M. Gonze et al. titled "Combining acoustic telemetry with archival tagging to investigate the spatial dynamics of the understudied pollack *Pollachius pollachius*", accepted for publication in the Journal of Fish Biology.

We will use the biologging tag "A19124", which was attached to pollock fish.

As for the reference Earth Observation (EO) data, we consider the European Union Copernicus Marine Service Information (CMEMS) product "NORTHWESTSHELF_ANALYSIS_FORECAST_PHY_004_013".

_NB: In addition to the Data Storage Tag (DST), the biologging data includes **teledetection by acoustic signals**, as well as the release and recapture/death information of the fish._

Both the reference EO and the biologging data are publicly available, and the computations should be tractable for most standard laptops.

**Workflow.**

Let's first summarize the key steps for running the geolocation:

1. **Define the configuration:** define the required parameters for the analysis.
2. **Compare the reference data with the DST information:** compare the data from the reference model with the biologging data. 
3. **Regrid the comparison to HEALPix:** translate the comparison into a HEALPix grid to avoid spatial distortion.
4. **Construct the temporal emission matrix:** create a temporal emission probability distribution (_pdf_) from the transformed grid.
5. **Construct another emission matrix with the acoustic detections:** calculate a similar model to the previous one, using this time the acoustic teledetections.
6. **Combine and normalize the matrices:** merge and normalize the two _pdfs_.
7. **Estimate (or _fit_) the geolocation model:** determine the parameters of the model based on the normalized emission matrix.
8. **Compute the state probabilities and generate trajectories:** compute the fish's location probability distribution and generate subsequent trajectories.
9. **Visualization:** visualize the evolution of the spatial probabilities over time and export the video.

Throughout this tutorial, you will gain practical experience in setting up and executing a typical workflow using `pangeo-fish` such that you can then apply the tool with your use-case study.

## 1. Initialization and configuration definition

In this step, we prepare the execution of the analysis.
It includes:
- Installing the necessary packages.
- Importing the required libraries.
- Defining the parameters for the next stages of the workflow.
- Configuring the cluster for distributed computing.
    

Only if you don't have good GPU (>=7.5)

In [ ]:
# do this
import torch
import os

os.environ["CUDA_VISIBLE_DEVICES"] = ""

### Mandatory libraries

In [ ]:
from pint_xarray import unit_registry as ureg
import hvplot.xarray
import xarray as xr
import sys

sys.path.append("../")
import pangeo_fish
from pangeo_fish.xarray_update import Accessor
from pangeo_fish.helpers import multiplot_healpix, multi_map_with_synced_sliders

### Parameters

In [ ]:
# tag_name corresponds to the name of the biologging tag name (DST identification number),
# which is also a path for storing all the information for the specific fish tagged with tag_name.

tag_name = "A18831"
# tag_name = "A19124"

# tag_root specifies the root URL for tag data used for this computation.
tag_root = "https://data-taos.ifremer.fr/data_tmp/cleaned/tag/"
# tag_root = "donnees_importees"
# ref_url is the path to the reference model.
# When set to None, load_model fetches the CMEMS Global Ocean Physics
# Reanalysis (daily-mean, 1/12 deg, `cmems_mod_glo_phy_my_0.083deg_P1D-m`)
# directly via the copernicusmarine client.
# To use a different reference, pass a `.yaml` intake catalog or a
# `.parq` / `.json` kerchunk reference here.
ref_url = None

# scratch_root specifies the root directory for storing output files.
# storage_options specifies options for the filesystem storing output files.

## example for remote storage
scratch_root = "s3://destine-gfts-data-lake/demo"
storage_options = {
    "anon": False,
    "profile": "gfts",
    "client_kwargs": {
        "endpoint_url": "https://s3.gra.perf.cloud.ovh.net",
        "region_name": "gra",
    },
}
## example for using your local file system instead
scratch_root = "./test/level_10_hpres"
storage_options = None

# Default chunk value for time dimension.  This values depends on the configuration of your dask cluster.
chunk_time = 1

# Either to use a HEALPix grid (["cells"]) or a 2D grid (["x", "y"])
dims = ["cells"]

# bbox, bounding box, defines the latitude and longitude range for the analysis area.
# bbox = {"latitude": [46, 51], "longitude": [-8, -1]}
bbox = {"latitude": [46, 59], "longitude": [-14, 2]}
# relative_depth_threshold defines the acceptable fish depth relative to the maximum tag depth.
# It determines whether the fish can be considered to be in a certain location based on depth. If you are not using a bathy pdf (calculate later at choice) put 0.8
relative_depth_threshold = 0

# refinement level  defines the resolution of the healpix grid used for regridding.
refinement_level = 10  # int(log2(4098))

# min_vertices sets the minimum number of vertices for a valid transcription for regridding.
min_vertices = 1

# differences_std sets the standard deviation for scipy.stats.norm.pdf.
# It expresses the estimated certainty of the field of difference.
differences_std = 0.75
# initial_std sets the covariance for initial event.
# It shows the certainty of the initial area.
initial_std = 1e-5
# recapture_std sets the covariance for recapture event.
# It shows the certainty of the final recapture area if it is known.
recapture_std = 1e-4
# earth_radius defines the radius of the Earth used for distance calculations.
earth_radius = ureg.Quantity(6371, "km")
# maximum_speed sets the maximum allowable speed for the tagged fish.
maximum_speed = ureg.Quantity(60, "km / day")
# adjustment_factor adjusts parameters for a more fuzzy search.
# It will factor the allowed maximum displacement of the fish.
adjustment_factor = 5
# truncate sets the truncating factor for computed maximum allowed sigma for convolution process.
truncate = 4

# receiver_buffer sets the maximum allowed detection distance for acoustic receivers.
receiver_buffer = ureg.Quantity(1000, "m")


# tolerance describes the tolerance level of the search during the fitting/optimization of the geolocation.
# Smaller values will make the optimization iterate more
tolerance = 1e-6

# track_modes defines the modes for generating fish's trajectories.
track_modes = ["mean", "mode"]
# additional_track_quantities sets quantities to compute for tracks using moving pandas.
additional_track_quantities = ["speed", "distance"]


# time_step defines the time interval between each frame of the visualization
time_step = 3
ellipsoid = "sphere"

In [ ]:
# Define target root directories for storing analysis results.
target_root = f"{scratch_root}/{tag_name}"

# Defines default chunk size for optimization.
default_chunk = {"time": chunk_time, "lat": -1, "lon": -1}
default_chunk_dims = {"time": chunk_time}
default_chunk_dims.update({d: -1 for d in dims})

### Dask

In [ ]:
# Set up a local cluster for distributed computing.
from distributed import LocalCluster

cluster = LocalCluster()
client = cluster.get_client()
client

### Opening the biologging data
Now that everything is set up, we can start by loading the biologging data (or _tag_)

In [ ]:
from pangeo_fish.helpers import load_tag

tag, tag_log, time_slice = load_tag(
    tag_root=tag_root, tag_name=tag_name, storage_options=storage_options
)
tag

You can plot the time series of the DST with the function `plot_tag()`:

### Credentials for reference model
The cells below allow you to create a file where your credentials for copernicus will be stocked (avoid the pain to enter them each time you run the cell). It allows you to commit this notebook without having to remove any part.

If you open the notebook for the first time unraw the following cell and execute it
from getpass import getpass

This cell allows to read your credential in a text file and use them


In [ ]:
credentials = {}
with open("copernicus_credentials.txt") as f:
    for line in f:
        line = line.strip()
        if line and "=" in line:
            key, value = line.split("=", 1)
            credentials[key.strip()] = value.strip()

Username = credentials["username"]
Password = credentials["password"]

## 2. Compare the reference data with the DST logs (using copernicus)

### Open the reference data model
Mandatory for bathy

In [ ]:
from pangeo_fish.helpers import load_model, compute_diff

In [ ]:
reference_model = load_model(
    uri=ref_url,
    tag_log=tag_log,
    time_slice=time_slice,
    bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
    chunk_time=chunk_time,
    remote_options={},
    Username=Username,
    Password=Password,
)

### Compute the difference

In [ ]:
%%time
diff = compute_diff(
    reference_model=reference_model,
    tag_log=tag_log,
    relative_depth_threshold=relative_depth_threshold,
    chunk_time=chunk_time,
)[0]

This should not sum to zero

In [ ]:
%%time
diff["diff"].count(["lat", "lon"]).plot()
diff

In [ ]:
%%time
diff.to_zarr(
    f"{target_root}/diff.zarr", mode="w", storage_options=storage_options, zarr_format=2
)

## 3. HEALPix regridding

In this step, we regrid the data from above to HEALPix coordinates. HEALPix (Hierarchical Equal Area isoLatitude Pixelization) is a spherical pixelation scheme that divides the sphere into equal-area pixels, eliminating spatial distortion [see the HEALPix documentation](https://healpix.sourceforge.io/).

This is a complex process, composed of several steps such as defining the HEALPix grid, creating the target grid and computing interpolation weights

Fortunately though, `pangeo-fish` embarks high-level functions to do the work for us!

In [ ]:
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset,
    regrid_dataset_hpresample,
)

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
diff = open_diff_dataset(target_root=target_root, storage_options=storage_options)
diff

In [ ]:
reshaped = regrid_dataset_hpresample(
    ds=diff, refinement_level=refinement_level, min_vertices=min_vertices, dims=dims
)
reshaped

Let's plot the same chart as before to check that the HEALPix regridding hasn't changed the data

In [ ]:
reshaped["diff"].count(dims).plot()

In [ ]:
# Saves the result
reshaped.chunk(default_chunk_dims).to_zarr(
    f"{target_root}/diff-regridded.zarr",
    mode="w",
    consolidated=True,
    compute=True,
    storage_options=storage_options,
    zarr_format=2,
)

If you want to see the diff, you may have to install ipleaflet and ipwidget and some other extensions (they don't have compatibility problem and will be asked)

In [ ]:
reshaped.pdf_plot.plot_healpix(var="diff", refinement_level=refinement_level, alpha=0.8)

## 4. Compute the emission probability distribution

In this step, we use the comparison result from the step above to construct the emission probability matrix.

This comparison is essentially he differences between the temperature measured by the tag and the reference sea temperature. 

The emission probability matrix represents the likelihood of observing a specific temperature difference given the model parameters and configurations.

In [ ]:
from pangeo_fish.helpers import compute_emission_pdf

In [ ]:
# Open the previous dataset (only necessary if you resume the notebook from here)
differences = xr.open_dataset(
    f"{target_root}/diff-regridded.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

# ... and compute the emission matrices
emission_pdf = compute_emission_pdf(
    diff_ds=differences,
    events_ds=tag["tagging_events"].ds,
    differences_std=differences_std,
    initial_std=initial_std,
    recapture_std=recapture_std,
    dims=dims,
    chunk_time=chunk_time,
)[0]
emission_pdf

If you want to see the pdf

Whatever the temporal distribution looks like, they must **never** (i.e, at _any time step_) sum to 0.

How could we check that visually? You'd have guessed it by now: similarly as before!

In [ ]:
emission_pdf = emission_pdf.chunk(default_chunk_dims).persist()
emission_pdf["pdf"].count(dims).plot()

In [ ]:
# Save the dataset
emission_pdf.to_zarr(
    f"{target_root}/emission_diff.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
emission_pdf.attrs["sources"] = emission_pdf.attrs.get("sources", []) + [
    "temperature"
]  # this line allows to keep in memory what pdf had been calculated so far
emission_pdf.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

## 5. Compute a 2nd _pdf_ with the acoustic detections

In this step, the goal is to calculate another emission distribution, this time from the acoustic detections.
**As such, it requires the tag to include at least one detection.**

These additional probabilities will enhance the emission _pdf_ constructed in the previous step by incorporating information from acoustic telemetry.

_NB: we will merge and normalize the two pdfs in the next stage of the workflow._

In [ ]:
from pangeo_fish.helpers import compute_acoustic_pdf

In [ ]:
# Load the previous emission pdf and compute the emission probabilities based on acoustic detections
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
acoustic_pdf = compute_acoustic_pdf(
    emission_ds=emission_diff,
    tag=tag,
    receiver_buffer=receiver_buffer,
    chunk_time=chunk_time,
    dims=dims,
)[0]
acoustic_pdf = acoustic_pdf.persist()
acoustic_pdf

If you wonder how this emission matrix looks like, you can plot a combined plot of the detections and the probabilities:

In [ ]:
tag["acoustic"]["deployment_id"].hvplot.scatter(c="red", marker="x") * (
    acoustic_pdf["acoustic"] != 0
).sum(dim=dims).hvplot()

### Explanations
On the plot above, at detection times the number of counted values drop to a few value (`5` in this example).

These numbers correspond to the number of pixels that covers the detection area.

Therefore, such drop is expected, since at those times we know that the fish was detected around the acoustic receivers, and so it **can't** be elsewhere.

These sporadic detections will constraint a lot the geolocation model upon optimizing!

**The next cell is optional. It will save the acoustic emission distribution. It is not necessary (see the next step).**

In [ ]:
acoustic_pdf.to_zarr(
    f"{target_root}/acoustic.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
)

#### Merging

In [ ]:
## ouvrir le total actuel
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
if "acoustic" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = current_total.attrs.get("sources", []) + [
        "acoustic"
    ]

    merged_pdf = current_total.merge(
        acoustic_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf

## 5.2. Compute a 3rd pdf with bathy

In this step, the goal is to calculate another emission distribution, this time using a bathymetry like GEBCO, EMODNET...

These additional probabilities will enhance the emission pdf constructed in the previous step by incorporating information from bathymetry information.

/!\ you must have run and save at least once chapter 2

In [ ]:
if "reference_model" in locals():
    pass
else:
    from pangeo_fish.helpers import load_model

    reference_model = load_model(
        uri=ref_url,
        tag_log=tag_log,
        time_slice=time_slice,
        bbox=(bbox | {"max_depth": tag_log["pressure"].max()}),
        chunk_time=chunk_time,
        remote_options={},
        Username=Username,
        Password=Password,
    )

In [ ]:
emission_diff = xr.open_dataset(
    f"{target_root}/emission_diff.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
emission_diff

In [ ]:
import fsspec
import numpy as np
from pangeo_fish.cf import bounds_to_bins
from pangeo_fish.tags import adapt_model_time, reshape_by_bins, to_time_slice
from pangeo_fish.bathy import batch_compute_pdf_bathy
from pangeo_fish.bathy import (
    compute_healpix_histogram_region_bin_size,
)

# bathy_url is the path to the bathymetry file
bathy_url = "https://data-taos.ifremer.fr/global_bathy/GEBCO_2024.zarr"

nside = 2**refinement_level

Importation de la base de donnée de bathymétrie

In [ ]:
full_bathy = xr.open_dataset(
    "s3://gfts-reference-data/gebco_2024_new.zarr",
    engine="zarr",
    chunks={},
    storage_options={
        "profile": "gfts",
        "client_kwargs": {"endpoint_url": "https://s3.gra.perf.cloud.ovh.net"},
    },
).rename({"lat": "latitude", "lon": "longitude"})

subset_bathy = full_bathy.sel(
    {dim: slice(bounds[0] - 0.1, bounds[1] + 0.1) for dim, bounds in bbox.items()}
)
ds_histo = compute_healpix_histogram_region_bin_size(
    subset_bathy,
    nside=nside,
    max_depth_m=1000,  # <- profondeur max désirée en mètres
    depth_bin_size=16,  # <- largeur d’un bin en mètres
)

In [ ]:
from pangeo_fish.bathy import batch_compute_pdf_bathy

In [ ]:
reshaped_tag = reshape_by_bins(
    tag_log,
    dim="time",
    bins=(
        reference_model.cf.add_bounds(["time"], output_dim="bounds")
        .pipe(bounds_to_bins, bounds_dim="bounds")
        .get("time_bins")
    ),
    other_dim="obs",
).chunk({"time": chunk_time})

pdf_da_func = batch_compute_pdf_bathy(
    ds_histo,
    reshaped_tag,
    target_root,  # remplace target_root
    batch_size=10000,
)
sum_over_cells = pdf_da_func.sum(dim="cells", skipna=True)

bathy_pdf = pdf_da_func / sum_over_cells
bathy_pdf

Enregistre la pdf de bathy seule (si besoin est de la réutiliser indépendamment des autres)

In [ ]:
bathy_pdf.compute().to_zarr(
    f"{target_root}/bathy_pdf_{tag_name}.zarr",
    compute=True,
    mode="w",
    consolidated=True,
    zarr_version=2,
    storage_options=storage_options,
)

In [ ]:
# If you want to see the pdf
bathy_pdf.pdf_plot.plot_healpix("pdf_bathy", refinement_level)

The following cells merge the bathy pdf with other pdfs (you need to calculate at least one other pdf to do so)

In [ ]:
## Open the dataset with the other pdfs
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
if "pdf_bathy" in current_total.data_vars:
    pass
else:
    current_total = current_total.load()
    current_total.attrs["sources"] = list(
        set(current_total.attrs.get("sources", [])) | {"bathymétrie"}
    )
    merged_pdf = current_total.merge(
        bathy_pdf.drop_indexes(["cell_ids"]), compat="override"
    )

    merged_pdf.to_zarr(
        f"{target_root}/total_pdf.zarr",
        mode="w",
        consolidated=True,
        storage_options=storage_options,
        zarr_format=2,
    )
merged_pdf
import json

print(json.dumps(merged_pdf.attrs))  # ou emission_with_bathy_tide.attrs

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [(merged_pdf, ["acoustic", "pdf", "pdf_bathy"])], refinement_level=10
)

multi_map_with_synced_sliders(liste)

## 5.3. Compute a 4th pdf with tide
 Warning, the reference dataset is on a fixed bbox, so the pdf you obtain after merging is smaller and so, less precise (it will be fixed when using Moko dataset)

In [ ]:
from pangeo_fish.tide import *
import xarray as xr
import pandas as pd
from ipywidgets import interact, IntSlider
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import sparse

In [ ]:
ds = tag_log["pressure"]
df_resample = ds.resample(
    {"time": "10min"}
).first()  # Take the pressure every ten minutes
df_resample = df_resample.reset_index("time")
df = df_resample.to_dataframe()
df = df.reset_index()  # time is no longer an index
df["time"] = (
    tag_log["time"].resample({"time": "10min"}).first()
)  # <--- Add a column "time"

tag_test = tide_behav_extr(
    df=df,
    tagno=tag_name,
    tideFL=1,  # time in hour used to detect tide
    tideLV=[
        0.42,
        0.85,
        0.6,
    ],  # hand made parameter ==> see Pedersen work on cod in Atlkantic
    behavFL=16,  # time in hour used to detect tide (use for the detection of behavior)
    behavLV=[0.42, 0.70, 0.2],
    dt=1,
)

tag_test

To see the detection

In [ ]:
plot_tide(tag_test)

In [ ]:
tag_test.to_zarr(
    f"{target_root}/tide_behav.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

In [ ]:
tide_behav = xr.open_dataset(f"{target_root}/tide_behav.zarr", engine="zarr")
tide_behav["behav_found"].plot()

### Import Tide database

In [ ]:
data_360 = xr.open_dataset("donnees_importees/Subset_NS_EC_forTMD30_21_2_2025.nc")
data = convert_lon_360_to_180(data_360, bbox)

ds_pdf = tide_pdf(tide_behav, data)
ds_pdf["tide_found"].plot()

# Save in zarr
ds_pdf.to_zarr(
    f"{target_root}/tide_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

### regrid in healpix

In [ ]:
import xarray as xr
from pangeo_fish.helpers import (
    open_diff_dataset,
    regrid_dataset,
    regrid_dataset_hpresample,
)

LIK = xr.open_dataset(f"{target_root}/tide_pdf.zarr", engine="zarr", chunks={})
LIK = LIK.rename({"mask": "ocean_mask"})
LIK["ocean_mask"] = LIK["ocean_mask"].compute()

tide_found_saved = LIK["tide_found"]

%time
lon2d, lat2d = xr.broadcast(LIK.longitude, LIK.latitude)
lon2d = lon2d.transpose("latitude", "longitude")
lat2d = lat2d.transpose("latitude", "longitude")

LIK_prepped = LIK.rename({"latitude": "yi", "longitude": "xi"}).assign_coords(
    longitude=(("yi", "xi"), lon2d.values),
    latitude=(("yi", "xi"), lat2d.values),
)
reshaped_tide = regrid_dataset_hpresample(
    ds=LIK_prepped, refinement_level=refinement_level, ellipsoid=ellipsoid
)
reshaped_tide["tide_found"] = tide_found_saved
# --- Densify ocean_mask if sparse ---
var = reshaped_tide["ocean_mask"].compute()
if isinstance(var.data, sparse.SparseArray):
    reshaped_tide["ocean_mask"] = var.copy(data=var.data.todense())

# --- Daily resampling (before writing, reduce the data volume) ---
ds = reshaped_tide.resample(time="1D").mean(skipna=True)

# --- Chunking ---
pdf_chunked = ds.pdf.chunk({"time": 10, "cells": -1})
ds_full = ds.drop_vars("pdf").assign(pdf=pdf_chunked)
ds_full = ds_full.chunk({"time": 10})
for var in ds_full.variables:
    if "chunks" in ds_full[var].encoding:
        del ds_full[var].encoding["chunks"]

store_path = f"{target_root}/tide_pdf_healpix.zarr"


ds_full.to_zarr(store_path, mode="w", compute=False)

n_time = ds_full.sizes["time"]
step = 200
vars_sans_time = ["cell_ids", "latitude", "longitude", "resolution", "ocean_mask"]
for start in range(0, n_time, step):
    end = min(start + step, n_time)
    print(f"Écriture time[{start}:{end}] / {n_time}")
    chunk = ds_full.isel(time=slice(start, end)).drop_vars(vars_sans_time)
    chunk.to_zarr(store_path, mode="r+", region={"time": slice(start, end)})

coords_only = ds_full[vars_sans_time]
coords_only.to_zarr(store_path, mode="a")

print("DONE — daily resample + zarr")

In [ ]:
ds_full

In [ ]:
ds_full.pdf_plot.plot_healpix("pdf", refinement_level=refinement_level)

### Merge with other pdfs

In [ ]:
## open the dataset with the others pdfs
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)

## open the daily tide_pdf
daily = xr.open_dataset(
    f"{target_root}/tide_pdf_healpix.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total = current_total.load()
current_total

In [ ]:
# Common pixels selection
common_pix = daily.cell_ids.to_index().intersection(current_total.cell_ids.to_index())

# Common time selection
common_time = daily.time.to_index().intersection(current_total.time.to_index())

ds1_common = daily.where(daily.cell_ids.isin(common_pix).compute(), drop=True).sel(
    time=common_time
)

ds1_common = ds1_common.rename({"pdf": "pdf_tide"})

ds2_common = current_total.where(
    current_total.cell_ids.isin(common_pix).compute(), drop=True
).sel(time=common_time)


emission_with_bathy_tide = ds1_common.merge(ds2_common, compat="override", join="exact")
emission_with_bathy_tide.attrs = current_total.attrs.copy()
tide_mask = emission_with_bathy_tide["tide_found"]
emission_with_bathy_tide = emission_with_bathy_tide.drop_vars(
    ["tide_found", "ocean_mask"]
)

The following cell present an example of use of multiplot. It allows to see different pdfs of the same fish, if they have the same bbox and the same time span and frequency.

In [ ]:
liste = multiplot_healpix(
    [(emission_with_bathy_tide, ["pdf", "pdf_bathy", "pdf_tide"])],
    refinement_level=refinement_level,
)
liste

multi_map_with_synced_sliders(liste)

In [ ]:
emission_with_bathy_tide = emission_with_bathy_tide.chunk({"time": 3, "cells": -1})
emission_with_bathy_tide.attrs["sources"] = list(
    set(current_total.attrs.get("sources", [])) | {"pdf_tide"}
)

emission_with_bathy_tide.to_zarr(
    f"{target_root}/total_pdf.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
emission_with_bathy_tide

## 6. Combine and normalize the X distributions

The normalization ensures that the probabilities sum up to one for each time step. 

In [ ]:
from pangeo_fish.helpers import combine_pdfs
from pangeo_fish.helpers import normalize_pdf

In [ ]:
current_total = xr.open_dataset(
    f"{target_root}/total_pdf.zarr",
    engine="zarr",
    chunks={},
    storage_options=storage_options,
)
current_total

In [ ]:
emission_pdf = normalize_pdf(
    ds=current_total,
    chunks=default_chunk_dims,
    dims=dims,
    exclude=("initial", "final", "mask"),
    excluded_pdf=None,  # ("acoustic",)
)[0]

emission_pdf.to_zarr(
    f"{target_root}/combined.zarr",
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)
combined = emission_pdf

(
    emission_pdf.sum(dims).hvplot(
        width=500, ylim=(0, 2), title="Sum of the probabilities"
    )
    + emission_pdf.count(dims).hvplot(
        width=500, title="Number of none-zero probabilities"
    )
).opts(shared_axes=False)

_The sums should equal to `1`._

In [ ]:
combined["pdf"].dggs.decode(
    {"grid_name": "healpix", "level": refinement_level, "indexing_scheme": "nested"}
).compute().dggs.explore()

## 7. Estimate the model's parameters

It is now time to determine the parameters of the model based on the normalized emission matrix.

Precisely, is consists of finding the best `sigma`, which corresponds to the standard deviation of the Brownian motion that models the fish's movement between the time steps.  

To do so, in the following we:
1. Define the lower and upper bounds for `sigma`.  
2. Search for the best `sigma` with `optimize_pdf()`.
3. Save the results of the search (i.e., ` sigma`), along with any additional parameters used during the optimization, a human-readable `.json` file.  

In [ ]:
from pangeo_fish.helpers import optimize_pdf

In [ ]:
# Open the distributions
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
emission

In [ ]:
# Define the parameter's bounds and search for the best value
params = optimize_pdf(
    ds=emission,
    earth_radius=earth_radius,
    adjustment_factor=adjustment_factor,
    truncate=truncate,
    maximum_speed=ureg.Quantity(60, "km / day"),
    tolerance=tolerance,
    dims=dims,
    # the results can be directly saved
    save_parameters=True,
    storage_options=storage_options,
    target_root=f"{target_root}",
    conv_method="FoscatConv",  # there you can chose other conv method (try something and see)
)
params

## 7 bis. Estimate the model's multiparameter with tide_behav 
(still experimental and does not works a hundred percent)\
 you need to run chapter 5.3

## 7 ter. Impose the model's multiparameter
If you don't have the time or if you want to try other values of sigma\
Use `create_parameter` to set a value of sigma, you can also set different value of sigma in a single tag.\
If you want to test different sigma and compare the result create different folders like in the following cells

In [ ]:
import json
import warnings
from pathlib import Path
import fsspec
from pangeo_fish.helpers import create_parameters, test_parameter
from pangeo_fish.helpers import optimize_pdf

First create the folder

In [ ]:
# Open the distributions
emission = xr.open_dataset(
    f"{target_root}/combined.zarr",
    engine="zarr",
    chunks=default_chunk_dims,
    inline_array=True,
    storage_options=storage_options,
)
# Create a new folder
emission.to_zarr(
    f"{target_root}/hand_sigma/combined.zarr",  # replace "hand_sigma" by another name
    mode="w",
    consolidated=True,
    storage_options=storage_options,
    zarr_format=2,
)

Then add the parameter

In [ ]:
params = create_parameters(
    sigma=[0.004708796991645453],
    sigma_indices=list(range(0, 57)),
    class_name="UpDownGaussian1DHealpix",
    target_root=f"{target_root}/hand_sigma",  # stayconsistent here
    save_parameters=True,
)

You can go faster using `test_parameter`. This function do the same as the three cells above. You can put an emission you opened in `emission = None ` or let it at None. If you let it at None it will open the combined pdf which was created in chapter 6.

In [ ]:
params = test_parameter(
    emission=None,
    sigma_tested=[0.001],
    sigma_indices=None,
    Conv_method="UpDownGaussian1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_UDG1D",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)
# Example if you want to try another value of sigma
params = test_parameter(
    emission=None,
    sigma_tested=[0.006],
    sigma_indices=None,
    Conv_method="Gaussian1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_G1D",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

# Example if you want to try another value of sigma
params = test_parameter(
    emission=None,
    sigma_tested=[0.006],
    sigma_indices=None,
    Conv_method="Foscat1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_Foscat",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

params = test_parameter(
    emission=None,
    sigma_tested=[0.004],
    sigma_indices=None,
    Conv_method="Foscat1DHealpix",
    target_root=target_root,
    saving_root="/hand_sigma_Foscat2",
    default_chunk_dims=default_chunk_dims,
    storage_options=storage_options,
)

# params=test_parameter(emission = None,
#                 sigma_tested = [0.004,0.02],
#                    sigma_indices = [list(range(0,25)),list(range(25,57))],
#                    Conv_method  = "UpDownGaussian1DHealpix",
#                    target_root = target_root,
#                    saving_root = "/hand_sigma_5",
#                    default_chunk_dims = default_chunk_dims,
#                    storage_options = storage_options   )

## 8. State probabilities and Trajectories

In this second to last step, we calculate the spatial probability distribution (based on the `sigma` found earlier) and further compute trajectories.

_NB: the computation precisely relies on `sigma` and the combined emission pdf._

In [ ]:
from pangeo_fish.helpers import predict_positions

In [ ]:
states, trajectories = predict_positions(
    target_root=f"{target_root}",
    storage_options=storage_options,
    chunks=default_chunk_dims,
    track_modes=track_modes,
    additional_track_quantities=additional_track_quantities,
    save=True,
)

the following cell is in the case you have created another sigma\
You can loop the process

In [ ]:
states_list = [
    predict_positions(
        target_root=f"{target_root}/{subfolder}",
        storage_options=storage_options,
        chunks=default_chunk_dims,
        track_modes=track_modes,
        additional_track_quantities=additional_track_quantities,
        save=True,
    )[0]
    for subfolder in [
        "hand_sigma",
        "hand_sigma_Foscat",
        "hand_sigma_Foscat2",
        "hand_sigma_G1D",
        "hand_sigma_UDG1D",
    ]
]

Let's quickly check that the positional probability distribution `states` never sums to 0 for all timesteps!

In [ ]:
(
    states.sum(dims).hvplot(width=500, ylim=(0, 2), title="Sum of the probabilities")
    + states.count(dims).hvplot(width=500, title="Number of none-zero probabilities")
).opts(shared_axes=False)

If you want to compare the states

In [ ]:
from pangeo_fish.helpers import multi_map_with_synced_sliders, multiplot_healpix

liste = multiplot_healpix(
    [
        (states_list[0], ["states"]),
        (states_list[1], ["states"]),
        (states_list[2], ["states"]),
        (states_list[4], ["states"]),
        (states_list[5], ["states"]),
    ],
    refinement_level=refinement_level,
)
liste

multi_map_with_synced_sliders(liste)

## 9. Visualization

In this final step, we visualize various aspects of the analysis results to gain insights and interpret the model outcomes. 

We plot the emission distribution, which represents the likelihood of observing a specific temperature difference given the model parameters and configurations. 

Additionally, we visualize the state probabilities, showing the likelihood of the system (i.e, the fish) being in different states (i.e, positions) at each time step. 

We also plot the trajectories decoded before (if you saved them).

They display the possible movement patterns over time. 

Finally, we render the emission matrix and state probabilities in a video and store it.

### 9.1 Plotting the trajectories 

In [ ]:
from pangeo_fish.helpers import plot_trajectories

In [ ]:
traj_plots = [
    plot_trajectories(
        target_root=f"{target_root}/{subfolder}",
        track_modes=track_modes,
        storage_options=storage_options,
        save_html=True,
    ).options(title=f"Model's Folder: {subfolder}")
    for subfolder in [
        "hand_sigma",
        "hand_sigma_Foscat",
        "hand_sigma_Foscat2",
        "hand_sigma_G1D",
        "hand_sigma_UDG1D",
    ]
]

In [ ]:
(traj_plots[0] + traj_plots[1] + traj_plots[2] + traj_plots[3] + traj_plots[4]).cols(2)

### 9.2 Plotting the `states` and `emission` distributions 

In [ ]:
from pangeo_fish.helpers import open_distributions, render_distributions

In [ ]:
data = open_distributions(
    target_root=target_root,
    storage_options=storage_options,
    chunks=default_chunk_dims,
    chunk_time=chunk_time,
)
data

The interactive plot above is too large to be stored as a `HMTL` file (as done earlier with the trajectories).

Fortunately, `pangeo-fish` can efficiently render images of `data` and build a video from them! 

In [ ]:
%pip install imageio[ffmpeg]

In [ ]:
video_filename = render_distributions(
    data=data,
    output_path=f"{target_root}/states",
    xlim=bbox["longitude"],
    ylim=bbox["latitude"],
    time_step=time_step,
    extension="mp4",
    frames_dir="images",
    remove_frames=True,
    storage_options=storage_options,
)